# Bring your own: a pack and an engine

> **Demonstration only:** scaffolding uses a temporary frozen fixture. It is not legal advice, a compliance verdict, or certification.

The CLI keeps TODOs and trusted-code warnings visible. This notebook only demonstrates the contributor path; the contracts belong to [`docs/authoring-packs.md`](../docs/authoring-packs.md) and [`docs/authoring-engines.md`](../docs/authoring-engines.md).

In [1]:
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

root = Path(tempfile.mkdtemp(prefix="reasonsmith-contributor-"))
# Resolve the console script from the interpreter running this notebook.  This avoids
# Windows PATH refresh and shell-quoting differences after the editable installs.
reasonsmith_executable = shutil.which("reasonsmith")
if reasonsmith_executable is None:
    candidates = [
        Path(sys.executable).with_name("reasonsmith.exe"),
        Path(sys.executable).parent / "Scripts" / "reasonsmith.exe",
        Path(sys.executable).with_name("reasonsmith"),
    ]
    reasonsmith_executable = next(str(path) for path in candidates if path.exists())

def cli(*args, cwd=root, check=False):
    result = subprocess.run([reasonsmith_executable, *args], cwd=cwd, text=True,
                            capture_output=True, check=check)
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    return result

cli("init", "pack", "demo_pack")
cli("init", "engine", "demo_engine")
print("pack TODO excerpt:")
print((root / "demo_pack/src/demo_pack/pack.toml").read_text().splitlines()[0])
print("engine warning excerpt:")
print((root / "demo_engine/src/demo_engine/engine.py").read_text().splitlines()[0])


Created pack scaffold in demo_pack
Created engine scaffold in demo_engine
pack TODO excerpt:
# TODO: replace every placeholder with a source-backed requirement
engine warning excerpt:
"""A declining engine scaffold; implement only after


The generated pack is intentionally invalid until its source-backed fields are edited. Here we replace placeholders with a tiny local fixture solely to exercise validation; in a real contribution, use reviewed source text and follow the authoring guide.

In [2]:
pack_file = root / "demo_pack/src/demo_pack/pack.toml"
text = pack_file.read_text()
replacements = {
    "TODO: pack title": "Demo pack",
    "TODO: describe the source, scope, and formalisation limits.": (
        "Frozen demonstration requirement."
    ),
    "TODO: official source or internal policy": "Demo source",
    "TODO: publication or provenance record": "Local fixture",
    "TODO: source document": "Demo source",
    "TODO: exact article or section": "section 1",
    "TODO: replace with the exact source text.": "The system must emit a decision record.",
    "TODO: affected stakeholder": "developer",
    "TODO: explain what this property establishes and what it leaves out.": "Checks presence only.",
    "todo_signal": "artifact_logs_decision_record",
}
for old, new in replacements.items():
    text = text.replace(old, new)
pack_file.write_text(text)
_ = cli("validate-pack", "--analyse", "demo_pack/src/demo_pack/pack.toml", cwd=root, check=True)


pack: demo_pack
title: Demo pack
description: Frozen demonstration requirement.
source.document: Demo source
source.publication: Local fixture
source.url: https://example.invalid/replace-me
requirements: 1
  demo_pack_todo | Demo source section 1 | record | binding: true | scope: unset | domains: none
analysis: demo_pack
  satisfiability: the encodable requirements are jointly satisfiable — some record discharges all of them at once
  entailment: no requirement entails another
  vacuity: no requirement is vacuously discharged on this evidence domain
  note: temporal decision procedure: not installed (BLACK solver binary not found on PATH). Install BLACK from your system package manager or https://www.black-sat.org. Nothing was answered from a weaker substitute.
  note: mutation coverage: no system was given, so no duty gets a score. Limit of this score: mutation analysis reaches only a system that exposes its decision logic as a rule block through sut.logic(), which is not most audited

Now install both editable projects. Installing a plug-in executes third-party packaging code: treat it as trusted code and review it before installation. Entry-point discovery is the mechanism; there is no hidden registry.

In [3]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(root / "demo_pack")],
    check=True, text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(root / "demo_engine")],
    check=True, text=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
_ = cli("validate-pack", "demo_pack", check=True)
verify = cli("verify-engine", "demo_engine")
assert verify.returncode == 0

# The same installed entry points are visible to Python, without another shell hop.
import importlib.metadata  # noqa: E402

import demo_engine  # noqa: E402
import demo_pack  # noqa: E402

pack_entry = next(ep for ep in importlib.metadata.entry_points(group="reasonsmith.packs")
                  if ep.name == "demo_pack")
engine_entry = next(ep for ep in importlib.metadata.entry_points(group="reasonsmith.engines")
                    if ep.name == "demo_engine")
print({"pack_entry_point": pack_entry.name, "pack_imported": demo_pack.__name__})
print({"engine_entry_point": engine_entry.name, "engine_type": type(demo_engine.engine).__name__})

# Keep the contributor's environment clean for the rest of the test suite.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "demo_pack", "demo_engine"],
    check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)


pack: demo_pack
title: Demo pack
description: Frozen demonstration requirement.
source.document: Demo source
source.publication: Local fixture
source.url: https://example.invalid/replace-me
requirements: 1
  demo_pack_todo | Demo source section 1 | record | binding: true | scope: unset | domains: none


engine: demo_engine
declared max_strength: observed

[PASS] triple 1: ecoa_reg_b_1002_9_a_1_timing_of_notice — expected satisfied/proved, got not_evaluated/None; witness: trusted-ceiling
      verdict match: no; strength within declared ceiling: yes
      declined
[PASS] triple 2: ecoa_reg_b_1002_9_b_2_specific_reasons — expected satisfied/proved, got not_evaluated/None; witness: trusted-ceiling
      verdict match: no; strength within declared ceiling: yes
      declined
[PASS] triple 3: ecoa_reg_b_1002_9_b_2_specific_reasons — expected satisfied/probed, got not_evaluated/None; witness: trusted-ceiling
      verdict match: no; strength within declared ceiling: yes
      declined
[PASS] triple 4: ecoa_reg_b_1002_9_b_2_specific_reasons — expected satisfied/observed, got not_evaluated/None; witness: trusted-ceiling
      verdict match: no; strength within declared ceiling: yes
      declined
[PASS] triple 5: ecoa_reg_b_1002_9_b_2_principal_reasons_complete — expected violated/probed, got

CompletedProcess(args=['/home/eduard/.treehouse/reasonsmith-722474/4/reasonsmith/.venv/bin/python', '-m', 'pip', 'uninstall', '-y', 'demo_pack', 'demo_engine'], returncode=0)